In [ ]:
import os
import json
import boto3
import folium
import pandas as pd
import awswrangler as wr
import matplotlib.pyplot as plt

from pathlib import Path
from data_profiling import ProfileReport
from folium.plugins import MarkerCluster


In [ ]:
session = boto3.Session(
    profile_name=os.getenv("AWS_PROFILE", "cuy-loyalty"),
    region_name="us-east-2",
)
print(session.region_name)

REPORTS_DIR = Path("reports")
REPORTS_DIR.mkdir(exist_ok=True)

DATABASE = "cuy_loyalty_dev_silver"

In [ ]:
def extract_and_profile(table_name, reports_dir = REPORTS_DIR, database = DATABASE, session=session):
    df = wr.s3.read_parquet_table(database = database, table=table_name, boto3_session=session)
    etl_columns = ['ingest_date', '_source_file','_processed_at']
    df.drop(columns=etl_columns, inplace=True)
    
    profile = ProfileReport(df, title=f"{table_name} Silver EDA Report")
    profile.to_file(reports_dir / f"{table_name}_silver_eda.html")
    
    return df

### Table Branches

In [ ]:
df_branches= extract_and_profile('branches')

In [ ]:
# Athena returns DECIMAL as object (decimal.Decimal): cast coordinates to float
df_branches[["latitude", "longitude"]] = df_branches[["latitude", "longitude"]].astype(float)
print(df_branches.shape)

# One marker per branch row; skip rows without usable coordinates
valid = df_branches["latitude"].between(-90, 90) & df_branches["longitude"].between(-180, 180)
print(f"{valid.sum()} mapped / {(~valid).sum()} skipped (missing or out-of-range coordinates)")
geo = df_branches[valid]

m = folium.Map(tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}", attr="Esri", zoom_start=2)
cluster = MarkerCluster().add_to(m)

for r in geo.itertuples():
    folium.Marker(
        location=[r.latitude, r.longitude],
        tooltip=r.branch_name,
        popup=folium.Popup(
            f"<b>{r.branch_name}</b><br>id: {r.branch_id}<br>{r.city}, {r.country}"
            f"<br>type: {r.branch_type}<br>status: {r.branch_status}",
            max_width=300,
        ),
    ).add_to(cluster)

m.fit_bounds(geo[["latitude", "longitude"]].values.tolist())
m.save(REPORTS_DIR / "branches_silver_map.html")
### some of the branches are not correctly placed on the map, probably because of wrong coordinates in the source data. The map is zoomed out to show all branches, but some of them are far away from the rest.

In [ ]:
df_branches.query("branch_id == 'SUC-UHIX6KQ4'")
df_branches[df_branches["longitude"].between(-1,1)] ## out of america, probably wrong coordinates

### Table call_center_interactions

In [ ]:
df_call_center_interactions= extract_and_profile('call_center_interactions')

In [ ]:
day_diff = (pd.to_datetime(df_call_center_interactions.process_date) - df_call_center_interactions.interaction_date).dt.days
day_diff.describe()


In [ ]:
df_call_center_interactions.groupby(['contact_reason','reason_category']).size().sort_values(ascending=False).head(10)
## duplicated column, keep only one of them, drop the other one. The column 'contact_reason' is more descriptive than 'reason_category', so we will keep it and drop the other one.

In [ ]:
df_call_center_interactions['sentiment_score'] = pd.to_numeric(df_call_center_interactions.sentiment_score)

In [ ]:
df_call_center_interactions[df_call_center_interactions.duration_seconds.isna()].describe(include='all').T
print(df_call_center_interactions[df_call_center_interactions.duration_seconds.isna()].groupby(['interaction_type']).size())
with pd.option_context('display.max_columns', None):
    display(df_call_center_interactions[df_call_center_interactions.duration_seconds.isna()])
## The duration_seconds column has some missing values, but the interaction_type column is not missing. We can see that the missing values are mostly for the interaction_type 'email', which makes sense because emails don't have a duration. We can also see that there are some missing values for 'chat', which might be worth investigating further.

In [ ]:
df_call_center_interactions[df_call_center_interactions.wait_time_seconds.isna()].describe(include='all').T
print(df_call_center_interactions[df_call_center_interactions.wait_time_seconds.isna()].groupby(['interaction_type','channel']).size())

In [ ]:
df_call_center_interactions.groupby(['customer_detected_accent','agent_used_accent']).size().sort_values(ascending=False).head(10)

### Table: Call Transcriptions

In [ ]:
df_call_transcripts  = extract_and_profile('call_transcripts')

In [ ]:
with pd.option_context('display.max_columns', None):
    display(df_call_transcripts)

In [ ]:
df_call_transcripts['mentioned_entities'][0]

In [ ]:
def safe_loads(x):
    try:
        return json.loads(x)
    except (TypeError, ValueError):
        return {}
    
entities = pd.json_normalize(df_call_transcripts["mentioned_entities"].apply(safe_loads))
entities.describe(include='all')
# include this in the silver extraction ..

In [ ]:
df_call_transcript_entities = pd.concat([df_call_transcripts.drop(columns="mentioned_entities"), entities.add_prefix("ent_")], axis=1)

In [ ]:
df_call_transcript_entities[df_call_transcript_entities.duration_seconds.isna()].describe(include='all').T
with pd.option_context('display.max_columns', None):
    display(df_call_transcript_entities[df_call_transcript_entities.duration_seconds.isna()])

### Table: campaign_sends

In [ ]:
df_campaign_sends  = extract_and_profile('campaign_sends')

In [ ]:
df_campaign_sends.columns

In [ ]:
df_campaign_sends.groupby(['send_status','was_delivered','was_opened'], dropna=False).size()

In [ ]:
df_campaign_sends[df_campaign_sends.was_opened.isna()].query("send_status == 'Sent'").describe().T
#replace with false

In [ ]:
df_campaign_sends.groupby(['send_channel','was_clicked'], dropna=False)[['click_count', 'click_date']].count()
# should I replace the missing values in the was_clicked column with False? It seems that the missing values are mostly for the send_channel 'SMS', which makes sense because SMS messages don't have a click event. However, there are also some missing values for 'Email', which might be worth investigating further. For now, I will replace the missing values with False.

In [ ]:
df_campaign_sends.groupby(['subject','template_used'], dropna=False).size()

In [ ]:
with pd.option_context('display.max_rows', None):
    display(
        df_campaign_sends.assign(
        template_used_sn=df_campaign_sends.template_used.str[:-2].fillna('Unknown')
    ).groupby(['subject','template_used_sn'], dropna=False).size()
    )

In [ ]:
df_campaign_sends[df_campaign_sends.template_used.isna()].describe(include='all').T
with pd.option_context('display.max_columns', None):
    display(df_campaign_sends[df_campaign_sends.template_used.isna()])

### Table: complaints

In [ ]:
df_complaints = extract_and_profile('complaints')

In [ ]:
display(df_complaints.groupby(['category','subcategory'], dropna=False).size())
df_complaints[df_complaints.subcategory.isna()].groupby(['category','description'], dropna=False).size()

In [ ]:
df_complaints[df_complaints.currency.isna()].describe(include='all').T

In [ ]:
df_complaints['origin_interaction_id']
## remove from silver layer, because it is not a valid foreign key to the call_center_interactions table. The origin_interaction_id column is not a valid foreign key because it contains values that do not exist in the call_center_interactions table. For example, the value 'CUI-00000000' does not exist in the call_center_interactions table. Therefore, we will remove this column from the silver layer.

### Table: customers

In [ ]:
df_customers = extract_and_profile('customers')

In [ ]:
with pd.option_context('display.max_columns', None):
    display(df_customers.query("last_updated > '2026-06-17'"))
## last_updated column has some future dates, which is not expected. We will investigate this further.

In [ ]:
df_customers[df_customers.credit_score.isna()].describe(include='all').T

In [ ]:
df_customers.groupby(['country','detected_accent']).size()

In [ ]:
df_customers.groupby(['country'])['estimated_monthly_income'].describe(include='all').T
# convert the estimated_monthly_income to the local currency to USD to normalize the data

### Table: daily_exchange_rates

In [ ]:
df_daily_exchange_rates = extract_and_profile('daily_exchange_rates')

In [ ]:
df_daily_exchange_rates[df_daily_exchange_rates['date'] == df_daily_exchange_rates['date'][0]].to_markdown()

In [ ]:
df_daily_exchange_rates.groupby(['source_currency','target_currency']).size()

In [ ]:
baseline_dates = pd.DataFrame({'date':pd.date_range('2023-06-17', '2026-06-17', freq='D')})
df_daily_exchange_rates['date'] = pd.to_datetime(df_daily_exchange_rates['date'])
merged_df = pd.merge(baseline_dates, df_daily_exchange_rates, on="date", how="left")
merged_df[merged_df["exchange_rate"].isna()]

assert len(baseline_dates)*12 == df_daily_exchange_rates.shape[0]
# no gaps in daily exchange rates, but the checks will be included in the silver layer to ensure that the data is complete and accurate.

### Table: digital_events

In [ ]:
df_digital_events = extract_and_profile('digital_events')

In [ ]:
df_digital_events.to_parquet("digital_events.parquet")

In [ ]:
(df_digital_events.event_date - pd.to_datetime(df_digital_events.process_date)).describe()

In [ ]:
df_digital_events[df_digital_events.customer_id.isna()].describe(include='all').T
# remove null customer_id as it is essential for the application?

In [ ]:
df_digital_events[df_digital_events.browser.isna()].describe(include='all').T

In [ ]:
import re

# SemVer 2.0.0 official regex pattern with named groups
SEMVER_REGEX = re.compile(
    r"^(?P<major>0|[1-9]\d*)\."
    r"(?P<minor>0|[1-9]\d*)\."
    r"(?P<patch>0|[1-9]\d*)"
    r"(?:-(?P<prerelease>(?:0|[1-9]\d*|\d*[a-zA-Z-][0-9a-zA-Z-]*)(?:\.(?:0|[1-9]\d*|\d*[a-zA-Z-][0-9a-zA-Z-]*))*))?"
    r"(?:\+(?P<buildmetadata>[0-9a-zA-Z-]+(?:\.[0-9a-zA-Z-]+)*))?$"
)


In [ ]:
df_digital_events['valid_app_version'] = df_digital_events['app_version'].apply(lambda x: bool(SEMVER_REGEX.match(x)) if pd.notnull(x) else False)
df_digital_events[~(df_digital_events.app_version.isnull())].valid_app_version.value_counts()

### Table: marketing_campaigns

In [ ]:
df_marketing_campaigns = extract_and_profile('marketing_campaigns')

In [ ]:
df_marketing_campaigns.groupby('description',dropna=False).size().sort_values(ascending=False)

In [ ]:
df_marketing_campaigns[df_marketing_campaigns.description.isna()].groupby(['campaign_objective','promoted_product'], dropna=False).size()
# description can be restored using the campaign_objective and promoted_product columns, which are not null. We can use these columns to fill in the missing description values. For example, if the campaign_objective is "Increase Sales" and the promoted_product is "Product A", we can fill in the description as "Increase Sales for Product A". We will create a mapping of (campaign_objective, promoted_product) to description and use it to fill in the missing values.

In [ ]:
len(set(list(map(lambda x: x[-1], df_marketing_campaigns.campaign_name.str.split('_')))))

In [ ]:
df_marketing_campaigns.campaign_name.str.split('_')

In [ ]:
df_marketing_campaigns[df_marketing_campaigns.promoted_product.isna()].head()
## also the promoted product can be restored by using the campaing_name ..
df_marketing_campaigns.assign(
    derived_product=df_marketing_campaigns.campaign_name.str.split('_').apply(lambda x: x[2])
).groupby(['promoted_product','derived_product']).size().to_markdown()

In [ ]:
df_marketing_campaigns.assign(
    derived_objective=df_marketing_campaigns.campaign_name.str.split('_').apply(lambda x: x[1])
).groupby(['campaign_objective','derived_objective'], dropna=False).size().to_markdown()

In [ ]:
df_marketing_campaigns[df_marketing_campaigns.target_segment.isna()].head()
## use target segment from customer table

In [ ]:
df_marketing_campaigns[df_marketing_campaigns.target_country.isna()].describe(include='all').T
## use country from customer table

In [ ]:
df_marketing_campaigns['budget_numeric'] = pd.to_numeric(df_marketing_campaigns.budget)
df_marketing_campaigns[df_marketing_campaigns.budget_numeric.isnull()].describe(include='all').T

In [ ]:
df_marketing_campaigns[df_marketing_campaigns.expected_conversion_rate.isnull()].head()

### Table: products

In [ ]:
df_products = extract_and_profile('products')

In [ ]:
df_products[df_products.duplicated(subset=['product_number'],keep=False)].sort_values(by='product_number')

In [ ]:
df_products['current_balance_numeric'] = pd.to_numeric(df_products['current_balance'])
df_products.groupby(['currency'])['current_balance_numeric'].describe().T

In [ ]:
df_products.groupby(['product_type','currency'])['current_balance_numeric'].mean().unstack()
## convert everything to USD

In [ ]:
df_products.groupby(['product_type','currency'])['credit_limit'].agg(['mean','count']).unstack()

In [ ]:
df_products[df_products['product_type'].isin(['Préstamo Hipotecario', 'Préstamo Personal','Tarjeta Crédito'])].groupby('product_type',dropna=False)['credit_limit'].describe()

In [ ]:
df_products.groupby(['product_type','currency'])['interest_rate'].agg(['mean','count']).unstack()

In [ ]:
df_products[df_products.interest_rate.isnull()].describe(include='all').T

In [ ]:
df_products[df_products.expiration_date.isna()].groupby('product_type').size()

In [ ]:
df_products.groupby('product_type')['days_past_due'].describe(include='all')

In [ ]:
df_products[df_products['product_type'].isin(['Préstamo Hipotecario', 'Préstamo Personal','Tarjeta Crédito'])].days_past_due.isna().sum()
## fill with cero ...

In [ ]:
df_products[df_products['product_type'].isin(['Préstamo Hipotecario', 'Préstamo Personal','Tarjeta Crédito'])].groupby('product_status')['days_past_due'].describe()

In [ ]:
df_products[(df_products['product_type'].isin(['Préstamo Hipotecario', 'Préstamo Personal','Tarjeta Crédito'])) & (df_products.days_past_due.isna())].groupby('product_status',dropna=False).size()

In [ ]:
df_products.isna().any()[df_products.isna().any()]

### Table: satisfaction_surveys

In [ ]:
df_satisfaction_surveys = extract_and_profile('satisfaction_surveys')

In [ ]:
df_satisfaction_surveys.groupby(['survey_type'],dropna=False)['main_score'].describe()

In [ ]:
(df_satisfaction_surveys
    .groupby(['survey_type', 'nps_category'], dropna=False)['main_score']
    .agg(count='count', mean='mean', std='std', min='min',
         q25=lambda s: s.quantile(.25), median='median',
         q75=lambda s: s.quantile(.75), max='max'))

In [ ]:
### impute missing nps categories only for survey_type = NPS
# Detractor: [0-6]
# Passive: [7-8]
# Promoters. [9 - 10]

### Table: service_agents

In [ ]:
df_service_agents = extract_and_profile('service_agents')

### Table: transactions

In [ ]:
df_transactions = extract_and_profile('transactions')

In [ ]:
(df_transactions.transaction_date - pd.to_datetime(df_transactions.process_date)).describe()

In [ ]:
df_transactions.groupby(['transaction_category','transaction_type'],dropna=False).size().unstack()

In [ ]:
df_transactions['amount_numeric'] = pd.to_numeric(df_transactions['amount'])
df_transactions.groupby(['transaction_type','transaction_category','currency'],dropna=False)['amount_numeric'].mean().unstack()

In [ ]:
df_transactions.groupby(['transaction_type','currency'])[['amount_numeric','amount_usd']].mean().unstack()
## use this to convert transaction amount to USD

In [ ]:
df_transactions.groupby(['transaction_category','merchant_category'],dropna=False).size().unstack()

In [ ]:
df_transactions.groupby(['merchant_category','transaction_type'],dropna=False).size().unstack()

In [ ]:
df_transactions.groupby(['transaction_country','transaction_city','currency'],dropna=False)['amount_numeric'].agg(['sum','count']).unstack()

In [ ]:
df_transactions.groupby('is_fraud')['fraud_score'].describe()

In [ ]:
from folium.plugins import HeatMap

#Athena returns DECIMAL as object (decimal.Decimal): cast coordinates to float
df_transactions[["latitude", "longitude"]] = df_transactions[["latitude", "longitude"]].astype(float)
df_transactions["amount_usd"] = pd.to_numeric(df_transactions["amount_usd"])
print(df_transactions.shape)

# Skip rows without usable coordinates
valid = df_transactions["latitude"].between(-90, 90) & df_transactions["longitude"].between(-180, 180)
print(f"{valid.sum()} mapped / {(~valid).sum()} skipped (missing or out-of-range coordinates)")
geo = df_transactions[valid]

# Every coordinate is unique (jittered), so plotting rows is not viable (~860MB html).
# Aggregate to one bubble per city; lat/lon std flags coordinates inconsistent with the city label.
by_city = (
    geo.dropna(subset=["transaction_city"])
    .groupby(["transaction_country", "transaction_city"])
    .agg(
        latitude=("latitude", "median"),
        longitude=("longitude", "median"),
        lat_std=("latitude", "std"),
        lon_std=("longitude", "std"),
        n_tx=("transaction_id", "size"),
        n_customers=("customer_id", "nunique"),
        amount_usd=("amount_usd", "sum"),
        fraud_rate=("is_fraud", "mean"),
        top_type=("transaction_type", lambda s: s.mode().iat[0] if s.notna().any() else None),
    )
    .reset_index()
)
display(by_city.sort_values("n_tx", ascending=False))

m = folium.Map(tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}", attr="Esri", zoom_start=2)

# Density layer: all mapped rows (incl. unknown city) binned to ~11km cells, weighted by count
bins = geo[["latitude", "longitude"]].round(1).value_counts().reset_index(name="n")
print(f"{len(bins)} heatmap cells")
HeatMap(bins[["latitude", "longitude", "n"]].values.tolist(), name="density (all mapped tx)", radius=12).add_to(m)

# City layer: bubble area proportional to transaction count
cities = folium.FeatureGroup(name="cities").add_to(m)
max_n = by_city["n_tx"].max()
for r in by_city.itertuples():
    folium.CircleMarker(
        location=[r.latitude, r.longitude],
        radius=5 + 25 * (r.n_tx / max_n) ** 0.5,
        fill=True,
        fill_opacity=0.6,
        tooltip=f"{r.transaction_city}: {r.n_tx:,} tx",
        popup=folium.Popup(
            f"<b>{r.transaction_city}, {r.transaction_country}</b><br>transactions: {r.n_tx:,}"
            f"<br>customers: {r.n_customers:,}<br>amount USD: {r.amount_usd:,.0f}"
            f"<br>fraud rate: {r.fraud_rate:.3%}<br>top type: {r.top_type}"
            f"<br>coord std (lat, lon): {r.lat_std:.2f}, {r.lon_std:.2f}",
            max_width=300,
        ),
    ).add_to(cities)

folium.LayerControl().add_to(m)
m.fit_bounds(by_city[["latitude", "longitude"]].values.tolist())
m.save(REPORTS_DIR / "transactions_silver_map.html")